# Week 13, Day 5: Optimisation in practice: early stopping, schedules and weight decay

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w013/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Stop at the right moment

Thirty noisy examples, 36 features. Run 5,000 steps of gradient descent from zero (step size n / largest eigenvalue of XᵀX) and record training and test mean squared errors after each step. Find `best_step`, and compare its test error with the best ridge test error over 60 values of λ.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2019)
n, p = 30, 36
w_true = rng.normal(0, 1, p) * np.exp(-np.arange(p) / 10)
X = rng.normal(0, 1, (n, p)); y = X @ w_true + rng.normal(size=n)
Xt = rng.normal(0, 1, (2000, p)); yt = Xt @ w_true + rng.normal(size=2000)

lr = n / np.linalg.norm(X, 2) ** 2
w = np.zeros(p)
train_err, test_err = [], []
for k in range(5000):
    ...
best_step = ...

lams = np.logspace(-3, 2, 60)
ridge_err = [np.mean((Xt @ np.linalg.solve(X.T @ X / n + l * np.eye(p), X.T @ y / n) - yt) ** 2) for l in lams]
best_ridge = min(ridge_err)
print(f"best step {best_step + 1}: test {min(test_err):.2f};  after 5000 steps: test {test_err[-1]:.2f};  best ridge {best_ridge:.2f}")
plt.semilogx(np.arange(1, 5001), train_err, label="train"); plt.semilogx(np.arange(1, 5001), test_err, label="test"); plt.legend(); plt.xlabel("step"); plt.show()

In [ ]:
import numpy as np
def test_curves():
    "training error ends near zero while test error ends higher than its minimum"
    assert train_err[-1] < 1e-6 and test_err[-1] > 1.5 * min(test_err)

def test_best_step():
    "best_step is the step with the lowest test error"
    assert test_err[best_step] == min(test_err)

def test_like_ridge():
    "early stopping's best test error is within 10% of the best ridge test error"
    assert abs(min(test_err) - best_ridge) / best_ridge < 0.1

test_curves()
test_best_step()
test_like_ridge()
print("All checks passed")

### Exercise: Constant, step and cosine

Run 4,000 steps of single-example SGD on a 10-weight linear regression with base learning rate 0.02 under three schedules, averaging the squared distance to the least-squares solution over five seeds. Store the results in the dictionary `errs`.

In [ ]:
rng = np.random.default_rng(2019)
N2, p2 = 2000, 10
X2 = rng.normal(0, 1, (N2, p2)); y2 = X2 @ rng.normal(0, 1, p2) + rng.normal(size=N2)
w_ls = np.linalg.lstsq(X2, y2, rcond=None)[0]
K, eta0 = 4000, 0.02

def sgd(schedule, seed):
    r = np.random.default_rng(seed); w = np.zeros(p2)
    for k in range(K):
        i = r.integers(N2)
        w = w - schedule(k) * (X2[i] @ w - y2[i]) * X2[i]
    return np.sum((w - w_ls) ** 2)

schedules = {
    "constant": lambda k: eta0,
    "step": ...,             # eta0 for the first half, eta0 / 10 after
    "cosine": ...,
}
errs = {name: np.mean([sgd(s, seed) for seed in range(5)]) for name, s in schedules.items()}
print({k: round(v, 4) for k, v in errs.items()})

In [ ]:
import numpy as np
def test_decay_helps():
    "both decaying schedules end at least five times closer to the least-squares solution than the constant one"
    assert errs["step"] * 5 < errs["constant"] and errs["cosine"] * 5 < errs["constant"]

test_decay_helps()
print("All checks passed")

### Exercise: L2 in Adam against decoupled decay

Two weights start at 1. The first gets no gradient from the data; the second gets large, zero-mean noise. Write `adam_l2(lam)` (penalty gradient added to g) and `adamw(lam)` (decay applied to w outside the Adam step), each running 500 steps with learning rate 0.001 and returning the final weights.

In [ ]:
def data_grads(steps, seed=0):
    r = np.random.default_rng(seed)
    return np.column_stack([np.zeros(steps), 100 * r.normal(size=steps)])

def adam_run(lam, decoupled, steps=500, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    w, m, v = np.ones(2), np.zeros(2), np.zeros(2)
    G = data_grads(steps)
    for k in range(1, steps + 1):
        g = G[k - 1] + (0 if decoupled else lam * w)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g ** 2
        w = w - lr * (m / (1 - b1 ** k)) / (np.sqrt(v / (1 - b2 ** k)) + eps)
        if decoupled:
            w = ...                       # decoupled weight decay
    return w

def adam_l2(lam): return adam_run(lam, decoupled=False)
def adamw(lam): return adam_run(lam, decoupled=True)
for lam in [0.01, 1.0]:
    print(f"lambda {lam}:  Adam + L2 {adam_l2(lam).round(3)}   AdamW {adamw(lam).round(3)}")

In [ ]:
import numpy as np
def test_l2_ignores_lambda():
    "with the L2 penalty inside Adam, the unused weight ends in the same place for lambda 0.01 and 1"
    assert abs(adam_l2(0.01)[0] - adam_l2(1.0)[0]) < 1e-3

def test_l2_spares_noisy():
    "with the L2 penalty inside Adam, the weight with large gradients barely decays even for lambda 1"
    assert adam_l2(1.0)[1] > 0.95

def test_adamw_uniform():
    "with AdamW and lambda 1, both weights decay to about exp(-0.5)"
    w = adamw(1.0)
    assert np.all(np.abs(w - np.exp(-0.5)) < 0.1)

test_l2_ignores_lambda()
test_l2_spares_noisy()
test_adamw_uniform()
print("All checks passed")